# 남해안 3개 여행지 검색 관심도 트렌드 분석

- 데이터: 네이버 데이터랩 검색어트렌드(주간, 상대 지수 0~100), `collect.py`로 수집
- 지역: 통영 / 거제 / 남해 (각각 `○○ 여행` + `○○여행` 합산)
- 이 노트북은 **API를 호출하지 않는다**. 저장된 `data/search_trend_weekly.csv`만 읽는다.
- 실행: 위에서부터 순서대로 (Cursor 상단 `Run All` 또는 셀마다 `Shift+Enter`)

## 단계 4-1. 데이터 불러오기

CSV의 날짜 열은 글자로 저장돼 있다. `parse_dates`로 **날짜 자료형**으로 바꿔 읽어야
"7일 간격인가", "몇 월인가" 같은 날짜 계산을 할 수 있다.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "data/search_trend_weekly.csv",
    index_col="period",      # 날짜 열을 표의 기준(행 이름)으로 사용
    parse_dates=["period"],  # 글자 '2021-01-04' → 날짜 자료형
    encoding="utf-8-sig",
)
df.head()

## 단계 4-2. 기본 정보 확인 (기간 · 컬럼 · 결측치)

과제 요구사항 Ⅱ-3의 세 가지를 차례로 확인한다.

In [ ]:
print("행 수(주):", len(df), " / 열:", list(df.columns))
print("기간:", df.index.min().date(), "~", df.index.max().date())
print()
print("[자료형] 숫자(float)여야 계산 가능")
print(df.dtypes)
print()
print("[결측치] 지역별 빈칸 수")
print(df.isna().sum())

### 날짜 빠짐 확인

결측치는 **빈칸**만 있는 게 아니다. 어떤 주가 **통째로 빠져 있으면** 빈칸조차 생기지 않는다.
그래서 이웃한 날짜끼리의 간격이 전부 7일인지 따로 확인한다.

In [ ]:
gaps = df.index.to_series().diff().dropna()   # 바로 앞 날짜와의 차이
print("날짜 간격 종류:", gaps.value_counts().to_dict())
print("모든 날짜가 월요일인가:", (df.index.dayofweek == 0).all())

### 기초 통계

- `mean`(평균)과 `50%`(중앙값)의 차이가 크면, 일부 큰 값이 평균을 끌어올리고 있다는 신호다.
- `std`(표준편차)는 값이 평균에서 평소 얼마나 벗어나는지를 나타낸다.

In [ ]:
df.describe().round(1)

## 단계 4-3. 경계 주 처리

- 데이터랩은 주간 값을 **월요일 시작 주**로 묶는다. 그래서 첫 주(2020-12-28)와 마지막 주(2025-12-29)는 요청 기간(2021-01-01~2025-12-31)과 일부만 겹친다.
- **결정: 262주 모두 유지.** 원본을 그대로 쓰고, 두 주의 값이 앞뒤 주와 비슷해 분석을 왜곡하지 않는다.
- 연도·월별로 묶을 때는 **주의 시작일(월요일)** 기준으로 분류한다. 예: 2020-12-28 주는 2020년 12월로 들어간다.

In [ ]:
df.iloc[[0, 1, -2, -1]]   # 처음 2주와 마지막 2주를 나란히 비교

## 단계 4-4. 이상치 탐지 (IQR 기준)

**IQR(사분위 범위)**: 값을 작은 순으로 줄 세웠을 때 가운데 50%가 차지하는 폭.
- Q1 = 하위 25% 지점, Q3 = 상위 25% 지점, IQR = Q3 − Q1
- **Q3 + 1.5×IQR 보다 크거나 Q1 − 1.5×IQR 보다 작으면** 이상치 후보로 본다.

평균·표준편차 대신 IQR을 쓰는 이유: 평균과 표준편차는 큰 값 자체에 끌려가 기준선이 같이 올라간다.
IQR은 가운데 절반만 보고 기준을 정해서 튀는 값의 영향을 덜 받는다.

In [ ]:
q1 = df.quantile(0.25)
q3 = df.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

pd.DataFrame({"하한": lower, "상한": upper}).round(1)

In [ ]:
# True/False 표: 해당 주·지역 값이 기준선을 벗어나면 True
is_outlier = (df > upper) | (df < lower)
print("지역별 이상치 후보 수:")
print(is_outlier.sum())

### 이상치 후보가 '오류'인가 '실제 사건'인가

값을 지우거나 고치기 전에, 후보들이 **언제** 몰려 있는지 본다.
특정 달에 매년 반복된다면 데이터 오류가 아니라 계절적인 실제 관심 급증일 가능성이 크다.

In [ ]:
# 이상치 후보를 (날짜, 지역, 값) 목록으로 펼친 뒤 월별로 개수 세기
outliers = df[is_outlier].stack().rename("ratio").reset_index()
outliers.columns = ["period", "region", "ratio"]
outliers["year"] = outliers["period"].dt.year
outliers["month"] = outliers["period"].dt.month

pd.crosstab(outliers["region"], outliers["month"])   # 행=지역, 열=월, 값=후보 개수

In [ ]:
pd.crosstab(outliers["region"], outliers["year"])    # 연도별로는 어떻게 흩어져 있나

### 처리 결정: 표시만 하고 값은 유지

- 판단 근거와 결론은 위 두 표의 결과를 보고 README 단계 4 기록에 **본인 말로** 적는다.
- 처리 방식: 값은 바꾸지 않는다. 이후 분석에서 쓸 수 있도록 "이상치 후보였는가"만 표시해 둔다.
- 버린 선택지: 제거(휴가철 데이터가 사라져 Q1·Q3의 답이 없어짐), 상한선으로 깎기(여름 최고점이 낮아져 계절성이 실제보다 약하게 보임)

In [ ]:
# 원래 값은 그대로 두고, 지역별 '이상치 후보 여부' 열만 옆에 붙인 사본을 만든다
flags = is_outlier.add_suffix("_이상치후보")
df_checked = df.join(flags)
df_checked[df_checked.filter(like="_이상치후보").any(axis=1)].head(10)

---
# 단계 5. 시계열 기법 ① 이동평균 + 시각화

**이동평균(moving average)**: 어떤 주의 값을 그 주 혼자가 아니라 **앞뒤 몇 주를 묶은 평균**으로 바꿔 보는 방법.
주 단위 값은 한 주만 특별한 일이 있어도 크게 흔들린다(노이즈). 이 흔들림을 눌러
"큰 흐름(추세)"이 보이게 만드는 것이 목적이다.

- 창(window) 12주 = 약 3개월. 계절이 바뀌는 흐름은 남기고, 한두 주짜리 요동은 지운다.
- `center=True`: 그 주를 **가운데** 두고 앞 6주·뒤 6주를 평균한다. 앞 12주만 쓰면 그래프가 실제보다 오른쪽으로 밀려 보인다.

## 5-1. 한글 폰트 설정

matplotlib은 기본 글꼴에 한글이 없어 축·범례의 한글이 네모(□)로 나온다.
컴퓨터에 있는 한글 글꼴을 지정해 두면 해결된다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 설치된 글꼴 중 한글 글꼴을 순서대로 찾아 첫 번째로 있는 것을 쓴다
installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if name in installed:
        matplotlib.rcParams["font.family"] = name
        break
matplotlib.rcParams["axes.unicode_minus"] = False  # 음수 기호가 깨지는 것 방지

print("사용 글꼴:", matplotlib.rcParams["font.family"])

## 5-2. 12주 이동평균 계산

`rolling(12)`은 "12주짜리 창을 한 주씩 밀면서 본다"는 뜻이고, `.mean()`이 그 창 안의 평균을 낸다.

In [ ]:
ma12 = df.rolling(window=12, center=True, min_periods=6).mean()

# 원본과 이동평균을 나란히 확인 (2022년 여름 앞뒤)
비교 = df.join(ma12.add_suffix("_12주평균")).loc["2022-07-04":"2022-08-08"]
비교.round(1)

## 5-3. 시각화 1 — 통영: 주간 원본 vs 12주 이동평균

한 지역만 놓고 "이동평균이 무엇을 하는지" 보여 주는 그림이다.
옅은 선이 원본(노이즈 포함), 진한 선이 12주 이동평균(추세)이다.

In [ ]:
COLORS = {"통영": "#3B5BDB", "거제": "#E8590C", "남해": "#099268"}

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(df.index, df["통영"], color=COLORS["통영"], alpha=0.28, linewidth=1.2, label="주간 원본")
ax.plot(ma12.index, ma12["통영"], color=COLORS["통영"], linewidth=2.2, label="12주 이동평균")

ax.set_title("통영 여행 검색 관심도 — 주간 원본과 12주 이동평균 (2021~2025)")
ax.set_ylabel("검색 관심도 (기간 내 최댓값=100)")
ax.set_xlabel("")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)          # 눈금선은 흐리게: 데이터가 주인공
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)  # 위·오른쪽 테두리 제거

fig.tight_layout()
fig.savefig("images/01_통영_이동평균.png", dpi=150)
plt.show()

## 5-4. 시각화 2 — 세 지역 12주 이동평균 비교

원본 3개를 겹쳐 그리면 선이 엉켜 읽히지 않는다. 이동평균만 그려 흐름을 비교한다.
선 끝에 지역 이름을 직접 붙여, 색만으로 구분하지 않아도 되게 한다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for region in df.columns:
    ax.plot(ma12.index, ma12[region], color=COLORS[region], linewidth=2.2, label=region)

# 선 끝에 지역 이름을 직접 붙인다. 끝값이 비슷하면 글자가 겹치므로,
# 값이 낮은 쪽부터 훑으면서 최소 간격(세로 폭의 4%)만큼 벌려 놓는다.
마지막 = ma12.dropna().iloc[-1].sort_values()
최소간격 = (ma12.max().max() - ma12.min().min()) * 0.04
직전y = None
for region, 값 in 마지막.items():
    y = 값 if 직전y is None else max(값, 직전y + 최소간격)
    ax.annotate(region, xy=(ma12.dropna().index[-1], y),
                xytext=(6, 0), textcoords="offset points",
                color=COLORS[region], fontweight="bold", va="center")
    직전y = y

ax.set_title("남해안 3개 여행지 검색 관심도 — 12주 이동평균 비교")
ax.set_ylabel("검색 관심도 (기간 내 최댓값=100)")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.25)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig("images/02_3개지역_이동평균.png", dpi=150)
plt.show()


## 5-5. 관찰 메모 (숫자로 확인)

그래프에서 눈으로 본 것을 숫자로 확인해 둔다. 해석(왜 그런가)은 단계 7에서 따로 쓴다.

주의: 아래 표의 **2020년 행은 주가 1개뿐**이다(경계 주 2020-12-28). 연도 비교에서는 제외하고 읽는다.

In [ ]:
연도별 = df.groupby(df.index.year).mean().round(1)
연도별.index.name = "연도"
연도별

---
# 단계 6. 시계열 기법 ②③ + 시각화 3·4

- **기법 ② 월별 집계**: 주간 값을 월 단위로 묶어 평균을 낸다. "연중 언제 높은가"(Q1)와 "겨울 비수기 격차"(Q4)에 답한다.
- **기법 ③ 변화율**: 전주 대비 몇 % 움직였는지 계산한다. "언제 갑자기 뛰었는가"(Q3)에 답한다.

각 주는 **주의 시작일(월요일)이 속한 달**로 분류한다(단계 4-3에서 정한 규칙).

## 6-1. 월별 집계 — 5년 평균 계절 패턴

`groupby`는 "같은 값끼리 묶어서 계산하라"는 뜻이다. 여기서는 월(1~12)로 묶어 평균을 낸다.
5년치를 한 번에 묶으므로, 특정 해의 사정보다 **반복되는 계절 패턴**이 드러난다.

In [ ]:
월별평균 = df.groupby(df.index.month).mean()
월별평균.index.name = "월"
월별평균.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for region in df.columns:
    ax.plot(월별평균.index, 월별평균[region], color=COLORS[region],
            linewidth=2.2, marker="o", markersize=5, label=region)

# 세 지역 모두 7월이 최고점이라 "최고 월" 표시는 한 번만 한다(같은 말을 3번 쓰지 않는다)
최고월들 = {region: int(월별평균[region].idxmax()) for region in df.columns}
if len(set(최고월들.values())) == 1:
    달 = next(iter(최고월들.values()))
    ax.annotate(f"세 지역 모두 {달}월 최고", xy=(달, 월별평균.max().max()),
                xytext=(10, -4), textcoords="offset points", ha="left",
                fontweight="bold", color="#343A40")
else:
    print("최고 월이 지역마다 다름:", 최고월들)

ax.set_title("월별 평균 검색 관심도 (2021~2025년 5년 평균)")
ax.set_ylabel("검색 관심도 (기간 내 최댓값=100)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels([f"{m}월" for m in range(1, 13)])
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="y", alpha=0.25)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig("images/03_월별_계절패턴.png", dpi=150)
plt.show()


## 6-2. 구간별 통계 — 여름 성수기 vs 겨울 비수기 (Q4)

- 여름 = 7·8월, 겨울 = 12·1·2월로 정한다(월별 그래프에서 봉우리와 골이 있는 구간).
- **비수기 비율 = 겨울 평균 ÷ 여름 평균**. 값이 작을수록 여름에 쏠려 있고 겨울에 더 많이 빠진다는 뜻이다.
- 비율로 보는 이유: 지역마다 값의 크기 자체가 다르므로(남해 33 vs 거제 11), 뺄셈으로 비교하면 큰 지역이 항상 불리하게 보인다.

In [ ]:
여름 = df[df.index.month.isin([7, 8])].mean()
겨울 = df[df.index.month.isin([12, 1, 2])].mean()

구간통계 = pd.DataFrame({
    "여름(7~8월) 평균": 여름.round(1),
    "겨울(12~2월) 평균": 겨울.round(1),
    "비수기 비율(겨울÷여름)": (겨울 / 여름).round(2),
})
구간통계

## 6-3. 변화율 — 전주 대비 급등 주 찾기 (Q3)

`pct_change()`는 바로 앞 주 대비 변화 비율을 계산한다. 0.25면 25% 상승이다.
급등 여부를 **비율**로 보는 이유: 값이 큰 지역일수록 같은 사건에도 숫자가 크게 움직이므로,
"몇 포인트 올랐나"로는 지역 간 비교가 안 된다.

In [ ]:
변화율 = df.pct_change() * 100   # % 단위

# 통영 기준 상위 급등 주 8개
급등 = 변화율["통영"].sort_values(ascending=False).head(8).round(1)
급등.to_frame("전주 대비 상승률(%)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
색 = ["#3B5BDB" if v >= 0 else "#ADB5BD" for v in 변화율["통영"]]
ax.bar(변화율.index, 변화율["통영"], width=5, color=색, linewidth=0)

# 상위 3개 주에만 날짜 표시 (모든 막대에 숫자를 붙이면 읽을 수 없다)
# 날짜가 가까우면 글자가 겹치므로 순위에 따라 위아래로 번갈아 띄운다
for 순위, (날짜, 값) in enumerate(변화율["통영"].nlargest(3).items()):
    ax.annotate(날짜.strftime("%Y-%m-%d"), xy=(날짜, 값), xytext=(0, 6 + 14 * (순위 % 2)),
                textcoords="offset points", ha="center", fontsize=9, color="#343A40")

ax.axhline(0, color="#868E96", linewidth=1)
ax.set_title("통영 — 전주 대비 검색 관심도 변화율 (주간)")
ax.set_ylabel("변화율 (%)")
ax.grid(axis="y", alpha=0.25)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig("images/04_통영_변화율.png", dpi=150)
plt.show()

## 6-4. 관찰 메모

- 위 표·그래프에서 읽은 **사실**만 적는다. 원인(왜)은 단계 7에서 관찰과 구분해 쓴다.
- 급등 주 날짜는 그 시기에 무슨 일이 있었는지(연휴·축제·방송 등) 따로 확인해야 해석할 수 있다. 확인 전에는 "원인 불명"으로 남겨 둔다.

In [ ]:
# 연도별 여름/겨울 비율의 변화도 함께 본다
연도구간 = pd.DataFrame({
    "여름": df[df.index.month.isin([7, 8])].groupby(df[df.index.month.isin([7, 8])].index.year).mean().stack(),
    "겨울": df[df.index.month.isin([12, 1, 2])].groupby(df[df.index.month.isin([12, 1, 2])].index.year).mean().stack(),
}).round(1)
연도구간["비율"] = (연도구간["겨울"] / 연도구간["여름"]).round(2)
연도구간.index.names = ["연도", "지역"]
연도구간.unstack()["비율"].dropna()

---
# 단계 9 (보너스). 시계열 분해 — 추세·계절성·나머지

하나의 선을 **세 조각으로 나눠** 보는 작업이다.

- **추세(trend)**: 몇 달 단위로 오르내리는 큰 흐름 → 이미 만든 12주 이동평균을 그대로 쓴다
- **계절성(seasonality)**: 매년 같은 시기에 반복되는 부분 → "원본 − 추세"를 주차별로 평균 낸 값
- **나머지(residual)**: 추세로도 계절성으로도 설명되지 않는 부분 → 원본 − 추세 − 계절성

원본 = 추세 + 계절성 + 나머지 로 다시 합쳐진다(덧셈 방식 분해).
나머지가 크게 튀는 주는 "평소 계절 패턴으로 설명 안 되는 사건이 있었던 주"다.

## 9-1. 세 조각 계산 (통영)

주차는 ISO 기준 주 번호(1~53)를 쓴다. 같은 주 번호끼리 모아 평균을 내면 "매년 그 시기의 평균적인 높낮이"가 나온다.

In [ ]:
지역 = "통영"

추세 = ma12[지역]                     # 12주 중심 이동평균 (단계 5에서 계산)
잔여 = df[지역] - 추세                # 추세를 뺀 나머지(계절성 + 불규칙)

주차 = df.index.isocalendar().week    # 각 날짜의 ISO 주 번호(1~53)
주차별평균 = 잔여.groupby(주차).mean()  # 매년 같은 주차끼리 평균 → 계절성
계절성 = pd.Series(주차.map(주차별평균).values, index=df.index)

나머지 = df[지역] - 추세 - 계절성

분해 = pd.DataFrame({"원본": df[지역], "추세": 추세, "계절성": 계절성, "나머지": 나머지})
분해.round(1).head()

In [ ]:
# 검산: 세 조각을 더하면 원본과 같아야 한다 (추세가 비어 있는 양 끝 제외)
차이 = (분해["추세"] + 분해["계절성"] + 분해["나머지"] - 분해["원본"]).abs().max()
print("원본과 (추세+계절성+나머지)의 최대 차이:", round(차이, 10))
print("계절성 진폭: 최저 %.1f (%d주차) ~ 최고 %.1f (%d주차)" % (
    주차별평균.min(), 주차별평균.idxmin(), 주차별평균.max(), 주차별평균.idxmax()))

## 9-2. 시각화 5 — 분해 결과 4단 그래프

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True)
항목 = [("원본", "#ADB5BD"), ("추세", COLORS[지역]), ("계절성", "#0B7285"), ("나머지", "#868E96")]

for ax, (이름, 색) in zip(axes, 항목):
    ax.plot(분해.index, 분해[이름], color=색, linewidth=1.6)
    ax.set_ylabel(이름)
    ax.grid(axis="y", alpha=0.25)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    if 이름 in ("계절성", "나머지"):
        ax.axhline(0, color="#CED4DA", linewidth=1)

axes[0].set_title(f"{지역} 검색 관심도 분해 — 원본 = 추세 + 계절성 + 나머지")

# 나머지가 가장 크게 튄 주 3개에 날짜 표시
for 순위, (날짜, 값) in enumerate(나머지.abs().nlargest(3).items()):
    axes[3].annotate(날짜.strftime("%Y-%m-%d"), xy=(날짜, 나머지[날짜]),
                     xytext=(0, 8), textcoords="offset points",  # 항상 위쪽에 두어 축 밖으로 나가지 않게 한다
                     ha="center", fontsize=9, color="#343A40")

fig.tight_layout()
fig.savefig("images/05_통영_시계열분해.png", dpi=150)
plt.show()

## 9-3. 해석 메모

- **계절성 진폭**: 위 출력의 "최저~최고"가 계절 하나만으로 설명되는 변동 폭이다.
- **나머지가 큰 주**: 계절 패턴으로 설명되지 않는 주다. 원인은 따로 확인해야 하며, 확인 전에는 "설명되지 않음"으로 남긴다.
- **한계**: 계절성을 5년 평균 하나로 고정했다. 해마다 계절 패턴 자체가 바뀌었다면 그 변화는 나머지로 밀려난다.